In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
import ast


In [2]:
ads_base = pd.read_csv('../data/ads.csv')
users_base = pd.read_csv('../data/users.csv')

users= users_base.copy()
ads = ads_base.copy()

In [3]:
users.head(1)

,user_id,age,gender,lat,lon,preferences
0,0,45,F,-33.106155,-53.58914,"[('Game_Hardware', 0.8485463411219264), ('Game..."


In [4]:
ads.head(1)

,ad_id,lat_target,lon_target,location_sensitive,max_radius_km,min_age,max_age,is_any_age,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate,click_probability
0,0,1.285495,-61.053986,0,2000,0,100,True,F,"{'Game_Development': 0.03083668354262722, 'For...",15.0,15000.0,0.988041,0.015604,0.031313


In [5]:
ads["categories"].head()

0    {'Game_Development': 0.03083668354262722, 'For...
1    {'MOBA': 0.08566344789189491, 'Mobile_Gaming':...
2    {'Sports_Games': 0.49102780454698103, 'Minecra...
3    {'Game_Hardware': 0.37218099077587796, 'Open_W...
4    {'MMORPG': 0.9785586009059007, 'PC_Gaming': 0....
Name: categories, dtype: object

In [6]:
type(users.preferences)

pandas.core.series.Series

In [7]:
# CSV perde a informação de tipo, então precisamos converter as strings de volta para listas e para dicionários. Para isso, podemos usar a função ast.literal_eval, que avalia uma string como uma expressão Python e retorna o objeto correspondente.

users["preferences"] = users["preferences"].apply(ast.literal_eval)
ads['categories'] = ads['categories'].apply(ast.literal_eval)

# Features Categorias

Gerar as categorias com seus pesos

OBS: isso foi feito porque para trabalhar com a abordagem de vetores, ADS e Users devem possuir mesma dimensões

Depois desenvolver uma forma mais inteligente para não fazer à mão

In [8]:
all_categories  = [
"FPS",
"Battle_Royale",
"MOBA",
"RPG",
"MMORPG",
"Strategy",
"Simulation",
"Sports_Games",
"Racing",
"Indie_Games",
"Retro_Gaming",
"Mobile_Gaming",
"PC_Gaming",
"Console_Gaming",
"Game_Development",
"Game_Streaming",
"Esports",
"Game_Hardware",
"VR_Gaming",
"Open_World",
"League_of_Legends",
"Fortnite",
"Minecraft",
"Counter_Strike",
"World_of_Warcraft"
]

In [9]:
# enumerar as categorias para criar um índice de categoria, que será usado para construir os vetores de características dos usuários e dos anúncios.
category_index = {cat: i for i, cat in enumerate(all_categories)}


In [10]:
category_index

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [11]:
# função para construir o vetor de características do usuário a partir de suas preferências, usando o índice de categoria para mapear as categorias para posições no vetor.

def build_user_vector(preferences, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in preferences:
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [12]:
# fazendo um teste para ver se a função build_user_vector está funcionando corretamente, construindo o vetor de características do primeiro usuário do dataset.
teste = build_user_vector(users.iloc[0]['preferences'], category_index)
display(teste)
display(users.iloc[0]['preferences'])
display(category_index)

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.15145366,
       0.        , 0.        , 0.84854634, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])

[('Game_Hardware', 0.8485463411219264),
 ('Game_Development', 0.15145365887807363)]

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [13]:
len(teste)

25

In [14]:
# função análoga para o ads
# feito isso pq o ads foi salvo como dict e user como list

def build_ad_vector(categories, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in categories.items():
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [15]:
# teste para ads
teste_ad = build_ad_vector(ads.iloc[0]['categories'], category_index)
display(teste_ad)
display(ads.iloc[0]['categories'])


array([0.        , 0.        , 0.        , 0.        , 0.35456129,
       0.        , 0.        , 0.01817338, 0.        , 0.03925901,
       0.        , 0.        , 0.        , 0.        , 0.03083668,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.17101861, 0.11956046, 0.        , 0.26659057, 0.        ])

{'Game_Development': 0.03083668354262722,
 'Fortnite': 0.11956046184830019,
 'Sports_Games': 0.018173378487163622,
 'MMORPG': 0.35456128766617023,
 'League_of_Legends': 0.17101861324341172,
 'Counter_Strike': 0.266590565443359,
 'Indie_Games': 0.03925900976896807}

In [16]:
# aplcar as funções para construir os vetores de características para todos os usuários e anúncios, e armazenar esses vetores em novas colunas nos dataframes.

users['feature_vector'] = users['preferences'].apply(lambda prefs: build_user_vector(prefs, category_index))
ads['feature_vector'] = ads['categories'].apply(lambda cats: build_ad_vector(cats, category_index))

In [17]:
users.head(1)

,user_id,age,gender,lat,lon,preferences,feature_vector
0,0,45,F,-33.106155,-53.58914,"[(Game_Hardware, 0.8485463411219264), (Game_De...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


# Gerar os vetores finais 

- vai fazer a similaridade tudo junto ou vai separar??

In [18]:
# definir ordem fixa das features

CATEGORY_COLUMNS = all_categories

In [19]:
# transformar features_vector em colunas separadas
category_features_users = pd.DataFrame(
    users["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

category_features_ads = pd.DataFrame(
    ads["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

users = pd.concat([users, category_features_users], axis=1)
ads = pd.concat([ads, category_features_ads], axis=1)

# Gerar os vetores finais sem cidade/estado

In [20]:
category_features_users

,FPS,Battle_Royale,MOBA,RPG,MMORPG,Strategy,Simulation,Sports_Games,Racing,Indie_Games,...,Game_Streaming,Esports,Game_Hardware,VR_Gaming,Open_World,League_of_Legends,Fortnite,Minecraft,Counter_Strike,World_of_Warcraft
0,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.848546,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.717441,0.000000,0.282559,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.040488,0.000000,0.000000,0.138106,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.149990,0.179896
4,0.000000,0.000000,0.0,0.319278,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.680722,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.000000,0.000000,0.0,0.199963,0.0,0.000,0.000000,0.000000,0.575985,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.007961
9996,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.537227,0.000000,0.000000
9997,0.156999,0.407886,0.0,0.000000,0.0,0.000,0.373462,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.061653,0.000000
9998,0.000000,0.574812,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.425188


In [21]:
users.iloc[0].preferences

[('Game_Hardware', 0.8485463411219264),
 ('Game_Development', 0.15145365887807363)]

In [22]:
users[category_features_users.columns].iloc[0]

FPS                  0.000000
Battle_Royale        0.000000
MOBA                 0.000000
RPG                  0.000000
MMORPG               0.000000
Strategy             0.000000
Simulation           0.000000
Sports_Games         0.000000
Racing               0.000000
Indie_Games          0.000000
Retro_Gaming         0.000000
Mobile_Gaming        0.000000
PC_Gaming            0.000000
Console_Gaming       0.000000
Game_Development     0.151454
Game_Streaming       0.000000
Esports              0.000000
Game_Hardware        0.848546
VR_Gaming            0.000000
Open_World           0.000000
League_of_Legends    0.000000
Fortnite             0.000000
Minecraft            0.000000
Counter_Strike       0.000000
World_of_Warcraft    0.000000
Name: 0, dtype: float64

In [23]:
FINAL_FEATURE_COLUMNS = (
    CATEGORY_COLUMNS
)

In [24]:
user_matrix = users[FINAL_FEATURE_COLUMNS].values
ad_matrix = ads[FINAL_FEATURE_COLUMNS].values

# PRECISA NORMALIZAR???

In [25]:
user_matrix

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.15699946, 0.40788586, 0.        , ..., 0.        , 0.06165252,
        0.        ],
       [0.        , 0.574812  , 0.        , ..., 0.        , 0.        ,
        0.425188  ],
       [0.00525721, 0.        , 0.        , ..., 0.        , 0.18760419,
        0.21895896]], shape=(10000, 25))

In [26]:
similarity_matrix = cosine_similarity(user_matrix, ad_matrix)

In [27]:
similarity_matrix
# cada linha representa um usuário e cada coluna representa um anúncio, e os valores são as similaridades entre eles.

array([[0.01098819, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.0634668 , 0.        , 0.6460098 , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.05764941, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.17108319, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(10000, 1000))

## Calcular o score

In [29]:
ads_base.head(1)

,ad_id,lat_target,lon_target,location_sensitive,max_radius_km,min_age,max_age,is_any_age,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate,click_probability
0,0,1.285495,-61.053986,0,2000,0,100,True,F,"{'Game_Development': 0.03083668354262722, 'For...",15.0,15000.0,0.988041,0.015604,0.031313


In [31]:
ads[['bid','budget','trust_score','estimated_conversion_rate','click_probability']].values

array([[1.50000000e+01, 1.50000000e+04, 9.88041352e-01, 1.56043122e-02,
        3.13130229e-02],
       [3.05496088e+01, 1.52748044e+04, 9.85822082e-01, 1.28880766e-02,
        2.40995844e-02],
       [1.50000000e+01, 1.50000000e+04, 8.05302758e-01, 1.67202380e-02,
        1.81406630e-02],
       ...,
       [2.08954711e+01, 2.08954711e+04, 7.67991304e-01, 2.47415589e-02,
        3.59687342e-02],
       [1.51880968e+01, 7.59404842e+03, 8.36877339e-01, 1.15583727e-02,
        4.23902320e-02],
       [2.05431338e+01, 2.05431338e+03, 7.79105317e-01, 3.68524652e-02,
        2.80041113e-02]], shape=(1000, 5))

## Gerar score distancia

In [32]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [ ]:
# fazer para tds os users
user_index = 50

user_lat = users.iloc[user_index]["lat"]
user_lon = users.iloc[user_index]["lon"]

ads["distance_km"] = haversine(
    user_lat,
    user_lon,
    ads["lat_target"],
    ads["lon_target"]
)

tau = 200

ads["distance_score"] = np.exp(-ads["distance_km"] / tau)

ads.loc[
    ads["location_sensitive"] == 0,
    "distance_score"
] = 1

In [ ]:
def compute_distance_score_matrix(users, ads, tau=200, return_df=True):
    """
    Calcula matriz (users x ads) de distance_score usando Haversine desenvolvido pelo Allan.

    """

    #1. Preparar arrays (broadcasting)
    user_lat = np.radians(users["lat"].values)[:, None]   # (U, 1)
    user_lon = np.radians(users["lon"].values)[:, None]

    ads_lat = np.radians(ads["lat_target"].values)[None, :]  # (1, A)
    ads_lon = np.radians(ads["lon_target"].values)[None, :]

    #2. Haversine vetorizado
    R = 6371

    dlat = ads_lat - user_lat
    dlon = ads_lon - user_lon

    a = np.sin(dlat / 2)**2 + np.cos(user_lat) * np.cos(ads_lat) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    distance_km = R * c   # (U, A)

    #3. Distance score
    distance_score = np.exp(-distance_km / tau)

    #4. Ajuste para ads não sensíveis
    mask = ads["location_sensitive"].values == 0
    distance_score[:, mask] = 1

    if return_df: # alterar isto aqui -> simplificar
        return pd.DataFrame(
            distance_score,
            index=users["user_id"].values,
            columns=ads["ad_id"].values
        )

    return distance_score

In [52]:
matrix_distance_score = compute_distance_score_matrix(users, ads, tau=200, return_df=True)

In [54]:
matrix_distance_score

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,1.0,1.0,1.767310e-04,1.0,9.942298e-10,1.0,1.0,2.057212e-05,1.0,1.111677e-07,...,1.225553e-08,9.218426e-09,1.154920e-04,4.290818e-04,5.791305e-08,7.859137e-05,2.731909e-04,1.705202e-02,1.069146e-09,1.273205e-05
1,1.0,1.0,3.561522e-04,1.0,2.477905e-05,1.0,1.0,1.138134e-04,1.0,8.712266e-04,...,3.037188e-06,2.094144e-05,8.273950e-03,2.083132e-02,4.376322e-04,7.351186e-02,5.615054e-07,7.852232e-05,7.361218e-06,3.750084e-04
2,1.0,1.0,7.663845e-05,1.0,9.824252e-09,1.0,1.0,3.905275e-04,1.0,2.388805e-07,...,2.294355e-07,1.475407e-07,1.593699e-03,8.337447e-04,7.319849e-07,6.944553e-04,1.231174e-03,2.526258e-01,1.509687e-08,2.355196e-04
3,1.0,1.0,1.869014e-05,1.0,1.938516e-04,1.0,1.0,3.303416e-07,1.0,8.149379e-02,...,1.347758e-07,3.735046e-06,1.526701e-05,1.212017e-04,1.033400e-04,1.174032e-04,7.404104e-10,1.042828e-07,6.934495e-06,1.437752e-06
4,1.0,1.0,5.949090e-11,1.0,1.256019e-04,1.0,1.0,3.678708e-06,1.0,5.122996e-09,...,6.362835e-03,7.787599e-03,8.239876e-07,4.506987e-09,3.077032e-04,4.311348e-07,1.964484e-09,4.622339e-09,7.079656e-03,8.892449e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1.0,1.0,8.552976e-08,1.0,1.756437e-02,1.0,1.0,1.312017e-07,1.0,3.197083e-04,...,9.915139e-07,4.740214e-05,1.864945e-06,1.262738e-06,5.382846e-04,8.027205e-06,7.741907e-11,4.769845e-09,2.904830e-04,6.546139e-07
9996,1.0,1.0,2.254846e-03,1.0,4.214485e-06,1.0,1.0,4.895339e-05,1.0,8.212521e-04,...,5.945518e-07,3.381537e-06,3.918921e-03,1.280513e-01,6.755135e-05,2.974955e-02,6.913321e-07,1.348927e-04,1.130575e-06,1.299245e-04
9997,1.0,1.0,5.640351e-06,1.0,3.587844e-04,1.0,1.0,9.681936e-04,1.0,6.276547e-05,...,1.841551e-04,1.347086e-03,1.481975e-02,4.174656e-04,2.081671e-02,2.889361e-02,7.083406e-07,3.020580e-05,3.308467e-04,4.616188e-03
9998,1.0,1.0,3.331295e-06,1.0,7.747231e-10,1.0,1.0,1.800568e-04,1.0,7.818122e-09,...,9.365534e-08,2.745988e-08,1.384447e-04,2.671984e-05,7.639530e-08,3.651926e-05,2.221697e-02,6.146832e-02,2.126813e-09,6.328747e-05


In [ ]:
# aqui está o motivo de ter mais distance_score 1 do que 0
ads.location_sensitive.value_counts()

location_sensitive
1    700
0    300
Name: count, dtype: int64

In [42]:
ads.distance_score.value_counts()

distance_score
1.000000e+00    300
1.301494e-08      1
2.564797e-04      1
5.148022e-07      1
1.368111e-04      1
               ... 
1.006110e-03      1
4.906674e-04      1
1.062906e-01      1
1.610480e-08      1
1.285630e-02      1
Name: count, Length: 701, dtype: int64

In [79]:
print(similarity_matrix.shape)
print(matrix_distance_score.shape)

(10000, 1000)
(10000, 1000)


## Gerar score final

similarity_matrix * w1 +
matrix_distance_score * w2 +
click * w3 +
trust * w4 +
bid * w5 + 
estimated_conversion_rate * w6

In [83]:
user_gender = users.iloc[user_index]["gender"]
user_age = users.iloc[user_index]["age"]

scores = similarity_matrix[user_index]

distance_scores = ads["distance_score"].values

final_scores = scores * distance_scores

ads_filtered = ads[
    (ads["gender_target"].isin([user_gender, "Any"])) &
    (
        (ads["is_any_age"]) |
        ((ads["min_age"] <= user_age) & (user_age <= ads["max_age"]))
    ) &
    (
        ads["distance_km"] <= ads["max_radius_km"]
    )
]
filtered_scores = final_scores[ads_filtered.index]

top_ads = ads_filtered.index[filtered_scores.argsort()[::-1][:100]]

In [ ]:
def compute_final_score(
    similarity_matrix,
    matrix_distance_score,
    ads,
    weights
):
    def normalize(x):
        return (x - x.min()) / (x.max() - x.min())

    click = normalize(ads["click_probability"].values)[None, :]
    trust = normalize(ads["trust_score"].values)[None, :]
    bid = normalize(ads["bid"].values)[None, :]
    estimated_conversion_rate = normalize(ads["estimated_conversion_rate"].values)[None, :]

    # --- pesos ---
    w1, w2, w3, w4, w5, w6 = weights

    # --- score final ---
    final = (
        similarity_matrix * w1 +
        matrix_distance_score * w2 +
        click * w3 +
        trust * w4 +
        bid * w5 + 
        estimated_conversion_rate * w6
    )

    return final

In [84]:
weights = [0.4, 0.2, 0.1, 0.1, 0.1,0.1]

final_scores = compute_final_score(
    similarity_matrix,
    matrix_distance_score,
    ads,
    weights
)

In [85]:
final_scores.shape

(10000, 1000)

In [69]:
ads.columns

Index(['ad_id', 'lat_target', 'lon_target', 'location_sensitive',
       'max_radius_km', 'min_age', 'max_age', 'is_any_age', 'gender_target',
       'categories', 'bid', 'budget', 'trust_score',
       'estimated_conversion_rate', 'click_probability', 'feature_vector',
       'FPS', 'Battle_Royale', 'MOBA', 'RPG', 'MMORPG', 'Strategy',
       'Simulation', 'Sports_Games', 'Racing', 'Indie_Games', 'Retro_Gaming',
       'Mobile_Gaming', 'PC_Gaming', 'Console_Gaming', 'Game_Development',
       'Game_Streaming', 'Esports', 'Game_Hardware', 'VR_Gaming', 'Open_World',
       'League_of_Legends', 'Fortnite', 'Minecraft', 'Counter_Strike',
       'World_of_Warcraft', 'distance_km', 'distance_score'],
      dtype='object')

## Gerar um filtro

OBS: seria mais interessante fazer isso antes de calcular similaridades

**Pensar em outra estrutura em vez de matrizes**

In [ ]:
def filter_and_rank(users, ads, final_scores):

    user_gender = users["gender"].values[:, None]
    user_age = users["age"].values[:, None]

    ad_gender = ads["gender_target"].values[None, :]
    ad_min_age = ads["min_age"].values[None, :]
    ad_max_age = ads["max_age"].values[None, :]
    ad_any_age = ads["is_any_age"].values[None, :]
    ad_radius = ads["max_radius_km"].values[None, :]
    ad_distance_km = ads['distance_km'].values[None, :]

    gender_mask = (ad_gender == "Any") | (ad_gender == user_gender)

    age_mask = (
        ad_any_age |
        ((ad_min_age <= user_age) & (user_age <= ad_max_age))
    )

    distance_mask = ad_distance_km <= ad_radius

    final_mask = gender_mask & age_mask & distance_mask

    scores = final_scores.copy()
    scores[~final_mask] = -np.inf


    return scores

In [ ]:
scores = filter_and_rank(users, ads, final_scores)

In [91]:
scores

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,0.382010,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,0.238174,-inf,-inf,-inf,-inf,-inf,-inf
1,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,0.104950,-inf,-inf,-inf,-inf,-inf,-inf,-inf
2,-inf,-inf,-inf,-inf,0.203293,-inf,-inf,-inf,-inf,-inf,...,-inf,0.181997,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
3,0.472677,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,0.371975,-inf,-inf,-inf,-inf,-inf,-inf
4,-inf,-inf,-inf,-inf,0.194696,-inf,-inf,-inf,-inf,-inf,...,-inf,0.183554,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.377614,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,0.238500,-inf,-inf,-inf,-inf,-inf,-inf
9996,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
9997,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,0.106259,-inf,-inf,-inf,-inf,-inf,0.174405,-inf
9998,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf


In [37]:
len(final_scores)

1000

In [482]:
len(filtered_scores)

112

In [483]:
top_ads

Index([968, 840, 933, 945, 782, 967, 528, 162, 486, 907, 949,  11, 842, 356,
       378, 284, 173, 444, 719, 607, 573, 728,  13, 118, 210, 707, 276, 918,
       105, 436, 675, 862, 226, 704, 816, 164, 651, 764, 993, 115, 605, 988,
       639, 320, 980, 609, 916, 383, 758, 788, 740, 657, 144, 132, 280, 694,
       172, 682, 213,  44,  52, 867, 410, 238, 411,  54,  32, 130,  77, 229,
       227,  37, 860, 218, 881, 937,  80, 979, 110, 884, 242, 157, 700, 826,
       844, 837, 735, 653, 636, 627, 745, 583, 572, 540, 537, 805, 474, 465,
       447, 445],
      dtype='int64')

In [484]:
scores[top_ads]

array([7.64812205e-01, 5.37067864e-01, 5.11222923e-01, 3.34307436e-01,
       2.06790161e-01, 1.34445248e-01, 5.08085209e-01, 1.16820124e-01,
       1.02267830e-01, 1.00062829e-01, 8.57175330e-02, 7.67727849e-02,
       6.23148859e-02, 6.12301688e-02, 5.62665130e-02, 3.51231829e-02,
       3.14774542e-02, 2.74046187e-02, 2.29743475e-02, 2.00039934e-02,
       9.13240795e-02, 1.16820124e-01, 7.92872444e-03, 7.01045744e-02,
       6.60347744e-03, 5.96724315e-01, 3.42187930e-01, 2.57266889e-01,
       1.40789311e-02, 2.41684732e-01, 7.64812205e-01, 7.63407823e-02,
       6.72953891e-02, 3.36127168e-01, 3.17076717e-04, 9.68327525e-03,
       4.13052419e-02, 9.56014523e-02, 9.23963252e-02, 2.78813746e-01,
       7.64812205e-01, 2.82142084e-01, 2.86017301e-01, 1.10108494e-01,
       2.81475206e-01, 3.22599375e-01, 3.95264494e-01, 2.90967620e-01,
       3.00200875e-02, 1.42255080e-01, 6.89795013e-02, 3.86312272e-01,
       1.32469773e-01, 2.62614853e-01, 1.16820124e-01, 2.20538705e-02,
      

In [485]:
ads_base.iloc[top_ads]

,ad_id,lat_target,lon_target,location_sensitive,max_radius_km,min_age,max_age,is_any_age,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate,click_probability
968,968,-28.723493,-47.699725,0,500,18,27,False,F,{'Game_Development': 1.0},30.622132,30622.131744,0.760771,0.020215,0.038003
840,840,-12.287043,-36.508292,0,5000,18,27,False,Any,"{'Game_Hardware': 0.07491083331498374, 'Battle...",15.000000,7500.000000,0.739663,0.019164,0.021376
933,933,-28.482272,-43.593784,0,2000,0,100,True,F,"{'Battle_Royale': 0.2251071979113922, 'RPG': 0...",24.558358,2455.835783,0.176868,0.007790,0.006216
945,945,-5.214401,-58.797164,0,5000,0,100,True,Any,"{'Fortnite': 0.34426096173802456, 'Console_Gam...",27.395168,13697.583943,0.709218,0.018249,0.018508
782,782,4.626985,-62.277855,0,5000,0,100,True,Any,"{'Battle_Royale': 0.36952502219661254, 'Esport...",15.000000,7500.000000,0.867341,0.045623,0.045357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,805,-26.478848,-44.978575,1,2000,0,100,True,Any,"{'Esports': 0.1893517086066517, 'Game_Streamin...",15.000000,7500.000000,0.312838,0.011044,0.004453
474,474,-29.336537,-69.225244,1,2000,0,100,True,F,{'Esports': 1.0},22.839631,2283.963139,0.767191,0.012949,0.031922
465,465,-3.953106,-49.613987,1,5000,0,100,True,F,{'PC_Gaming': 1.0},15.000000,1500.000000,0.909579,0.053659,0.032926
447,447,-33.425381,-44.178440,1,2000,0,100,True,F,"{'Open_World': 0.9801781271671006, 'PC_Gaming'...",22.632010,22632.009846,0.539597,0.032075,0.018062


In [486]:
users_base.loc[user_index]

user_id                                                       50
age                                                           23
gender                                                         F
lat                                                   -28.451879
lon                                                   -51.765924
preferences    [('Racing', 0.09784719095425211), ('Fortnite',...
Name: 50, dtype: object

# **OBJETIVO : score final**

score =
interest_similarity
× demographic_score
× bid
× trust_score
× estimated_conversion_rate

NÃO ESQUECER: incluir as